# Transformer Predictive Maintenance — Edge AI (ESP32)End-to-end TinyML pipeline: **data → train/dev/test → int8 quantisation → C header for an MCU**.| stage | what it does ||---|---|| `01_data` | ingest a real CSV *or* generate IEEE C57.91 physics-based data || `02_training` | multi-head 1-D CNN: health state **+** winding-temperature forecast || `03_evaluation` | metrics vs honest baselines, plots, markdown report || `04_deployment_esp32` | int8 TFLite, C array, feature parity tests, firmware |**Runtime:** ~3 minutes on a Colab CPU. No GPU needed — the model is 8 K parameters.

## 1 · SetupClone the repo and install dependencies.

In [ ]:
!git clone --depth 1 --branch arena/01a0b0f4-kalmanfilter-lab \    https://github.com/MohamedMehery/Kalmanfilter_lab.git repo 2>/dev/null || echo "already cloned"%cd repo/Transformer_PdM_EdgeAI!pip install -q tensorflow numpy pandas scikit-learn matplotlib

## 2 · Data**Using your own CSV?** Upload it, then point the loader at it. Column names aremapped automatically via `COLUMN_ALIASES` in `config.py` — `OTI`, `OilTemp`,`Top-oil` all resolve to the same canonical field. Anything missing is estimatedand reported.Otherwise the cell below synthesises data from the IEEE C57.91 thermal modelwith injected faults (overload, cooling loss, oil leak, unbalance).

In [ ]:
# Option A — your own data:# from google.colab import files; up = files.upload()# !python 01_data/preprocess.py --csv "$(ls *.csv | head -1)"# Option B — physics-based synthetic (default)!python 01_data/make_synthetic.py!python 01_data/preprocess.py

### Inspect the splitNote `persistence_acc` in the metadata. A transformer's health state barelychanges over a 2-hour horizon, so "assume nothing changes" already scores ~96%.**Any accuracy claim below that number is worthless** — this is the bar themodel has to clear, and it's why the evaluation leads with early-warning recallinstead of raw accuracy.

In [ ]:
import jsonmeta = json.load(open('01_data/splits/meta.json'))print("features :", meta['n_features'], "| window:", meta['window'], "| horizon:", meta['horizon'])print("split    :", meta['sequences'])print("balance  :", meta['class_balance'], "(NORMAL / WARNING / CRITICAL)")print("persistence baseline:", {k: round(v,4) for k,v in meta['persistence_acc'].items()})print("early-warning cases :", meta['early_warning'])

## 3 · TrainMulti-head CNN. Model selection uses dev-set **alarm-F1**, not `val_loss` (class weights make the loss unstable) and not accuracy (persistence-dominated).

In [ ]:
!python 02_training/train.py --epochs 150

## 4 · Quantise to int8 + emit the C headerFull-integer quantisation, then the whole test split is re-run through the interpreter to prove the int8 model still behaves.

In [ ]:
!python 04_deployment_esp32/scripts/convert_tflite.py

## 5 · Evaluate

In [ ]:
!python 03_evaluation/evaluate.py

In [ ]:
from IPython.display import Image, display, Markdowndisplay(Markdown(open('03_evaluation/report.md').read()))

In [ ]:
from IPython.display import Image, displayfor p in ['01_confusion','02_wti_forecast','03_risk_trace','04_alarm_curve','05_training','06_quant_fidelity']:    display(Image(f'03_evaluation/plots/{p}.png'))

## 6 · Verify the edge path *before* flashingTrain/serve skew is the classic silent TinyML failure: the firmware computesfeatures slightly differently from the training script, the model still returnsconfident numbers, and they're wrong. These two tests make that impossible tomiss — they compile the actual C code and diff it against Python.

In [ ]:
!python 04_deployment_esp32/scripts/test_parity.py      # C features == Python features!python 04_deployment_esp32/scripts/make_replay_data.py --n 240!python 04_deployment_esp32/scripts/verify_pipeline.py  # C pipeline + int8 == host decisions

## 7 · Download the firmware artifactsFlash with PlatformIO:```bashcd 04_deployment_esp32pio run -t upload && pio device monitor````APP_MODE 0` replays the bundled test rows so you can confirm the boardreproduces these exact numbers; set `APP_MODE 1` and wire up `read_sensors()`for live operation.

In [ ]:
import osprint("int8 model :", os.path.getsize('artifacts/model_int8.tflite'), "bytes")print("C header   :", os.path.getsize('04_deployment_esp32/include/transformer_pdm_model.h'), "bytes")# from google.colab import files# files.download('04_deployment_esp32/include/transformer_pdm_model.h')